In [1]:
import warnings
import re
import json
import pickle

import numpy as np
import pandas as pd

In [2]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

In [3]:
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')


In [4]:
from sklearn.preprocessing import (
    MultiLabelBinarizer,
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif   
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
import joblib

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

from sklearn.cluster import KMeans

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

In [7]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)


In [8]:
url = 'https://raw.githubusercontent.com/rashakil-ds/Public-Datasets/refs/heads/main/Bank%20Data.csv'
df = pd.read_csv(url)
df.head(3)

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance
0,0x160a,CUS_0xd40,September,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,7,11.27,2022.0,Good,809.98,35.030402,22 Years and 9 Months,No,49.574949,236.64268203272135,Low_spent_Small_value_payments,186.26670208571772
1,0x160b,CUS_0xd40,October,Aaron Maashoh,24,821-00-0265,Scientist,19114.12,1824.843333,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,9,13.27,4.0,Good,809.98,33.053114,22 Years and 10 Months,No,49.574949,21.465380264657146,High_spent_Medium_value_payments,361.44400385378196
2,0x160c,CUS_0xd40,November,Aaron Maashoh,24,821-00-0265,Scientist,19114.12,1824.843333,3,4,3,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",-1,4,12.27,4.0,Good,809.98,33.811894,NaN,No,49.574949,148.23393788500925,Low_spent_Medium_value_payments,264.67544623342997


In [9]:
# Checking for null values
df.isna().sum()

ID                             0
Customer_ID                    0
Month                          0
Name                        5015
Age                            0
SSN                            0
Occupation                     0
Annual_Income                  0
Monthly_Inhand_Salary       7498
Num_Bank_Accounts              0
Num_Credit_Card                0
Interest_Rate                  0
Num_of_Loan                    0
Type_of_Loan                5704
Delay_from_due_date            0
Num_of_Delayed_Payment      3498
Changed_Credit_Limit           0
Num_Credit_Inquiries        1035
Credit_Mix                     0
Outstanding_Debt               0
Credit_Utilization_Ratio       0
Credit_History_Age          4470
Payment_of_Min_Amount          0
Total_EMI_per_month            0
Amount_invested_monthly     2271
Payment_Behaviour              0
Monthly_Balance              562
dtype: int64

In [10]:
df[~df['Age'].astype(str).str.isnumeric()]['Age'].value_counts().to_frame()

,count
Age,
-500,464
32_,89
36_,82
19_,78
39_,77
...,...
6400_,1
4938_,1
6470_,1


In [11]:
df['Age'] = df['Age'].astype(str).str.rstrip('_') 
df['Age'] = df['Age'].astype(str).str.lstrip('-') 
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

In [12]:
x =df.groupby('Customer_ID')['Name']

In [13]:
df['Name'] = df.groupby('Customer_ID')['Name'].transform(lambda x: x.ffill().bfill())

In [14]:
df.loc[(df['Age'] < 10) | (df['Age'] > 60), 'Age'] = np.nan

df['Age'] = df.groupby('Customer_ID')['Age'].transform(lambda x: x.ffill().bfill())
df['Age'].fillna(df['Age'].median(), inplace=True)

In [15]:
df['SSN'].value_counts().to_frame()

,count
SSN,
#F%$D@*&8,2828
821-00-0265,4
756-60-6274,4
089-19-7596,4
905-14-5788,4
...,...
332-96-0334,1
611-21-4406,1
326-89-0421,1


In [16]:
df.drop(columns=['SSN'],inplace = True)

In [17]:
df['Occupation'].value_counts().to_frame()

,count
Occupation,
_______,3438
Lawyer,3324
Engineer,3212
Architect,3195
Mechanic,3168
Developer,3146
Accountant,3133
Media_Manager,3130
Scientist,3104


In [18]:
df['Occupation'] = df['Occupation'].replace('_______', np.nan)

df['Occupation'] = df.groupby('Customer_ID')['Occupation'].transform(lambda x: x.ffill().bfill())

In [19]:
occupation_counts = df['Occupation'].value_counts()
occupation_counts.to_frame().style.background_gradient()

,count
Occupation,
Lawyer,3548
Engineer,3432
Architect,3412
Mechanic,3388
Scientist,3372
Accountant,3372
Developer,3360
Media_Manager,3360
Teacher,3336


In [20]:
df['Annual_Income'] = df['Annual_Income'].astype(str).str.replace(r'[^0-9.]', '', regex=True)

df['Annual_Income'] = pd.to_numeric(df['Annual_Income'])

In [21]:
df['Annual_Income'].describe().round(0).astype(int).to_frame()

,Annual_Income
count,50000
mean,166334
std,1351965
min,7006
25%,19453
50%,37578
75%,72817
max,24137255


In [22]:
df['Monthly_Inhand_Salary'].describe().to_frame(name='Monthly Inhand Salary').round().astype(int)

,Monthly Inhand Salary
count,42502
mean,4182
std,3174
min,304
25%,1625
50%,3086
75%,5934
max,15205


In [23]:
df['Monthly_Inhand_Salary'] = df.groupby('Customer_ID')['Monthly_Inhand_Salary'].transform(lambda x: x.ffill().bfill())

df['Monthly_Inhand_Salary'].fillna(df['Monthly_Inhand_Salary'].median(), inplace=True)

In [24]:
df['Monthly_Inhand_Salary'].describe().to_frame(name='Monthly Inhand Salary Description').round().astype(int)

,Monthly Inhand Salary Description
count,50000
mean,4181
std,3173
min,304
25%,1624
50%,3082
75%,5936
max,15205


In [25]:
df.groupby('Occupation')['Monthly_Inhand_Salary']\
.mean()\
.sort_values(ascending=False)\
.to_frame(name='Monthly nhand Salary Mean')\
.round(0)\
.style.background_gradient(cmap='coolwarm')

,Monthly nhand Salary Mean
Occupation,
Architect,4302.000000
Manager,4249.000000
Musician,4248.000000
Scientist,4232.000000
Writer,4220.000000
Entrepreneur,4220.000000
Engineer,4198.000000
Accountant,4193.000000
Media_Manager,4193.000000


In [26]:
num_bank_acc_description = df['Num_Bank_Accounts'].describe().to_frame()
num_bank_acc_description

,Num_Bank_Accounts
count,50000.000000
mean,16.838260
std,116.396848
min,-1.000000
25%,3.000000
50%,6.000000
75%,7.000000
max,1798.000000


In [27]:
num_accounts_val_count = df['Num_Bank_Accounts'].value_counts().to_frame().sort_values(by='count', ascending=False)
print("Most 15 Number of Bank Accounts Repeated:")
num_accounts_val_count.nlargest(15, columns='count')

Most 15 Number of Bank Accounts Repeated:


,count
Num_Bank_Accounts,
6,6504
7,6408
8,6387
4,6100
5,6068
3,5955
9,2738
10,2599
1,2253


In [28]:
df.loc[(df['Num_Bank_Accounts'] <= 0)|(df['Num_Bank_Accounts'] > 20),'Num_Bank_Accounts'
] = np.nan

In [29]:
df['Num_Bank_Accounts'] = df.groupby('Customer_ID')['Num_Bank_Accounts'].transform(lambda x: x.ffill().bfill())
df['Num_Bank_Accounts'].fillna(df['Num_Bank_Accounts'].median(), inplace=True)

In [30]:
df['Num_Bank_Accounts'].describe().to_frame()

,Num_Bank_Accounts
count,50000.000000
mean,5.633000
std,2.326813
min,1.000000
25%,4.000000
50%,6.000000
75%,7.000000
max,11.000000


In [31]:
df['Num_Bank_Accounts'] = df['Num_Bank_Accounts'].astype(int)

In [32]:
df['Interest_Rate'].describe().to_frame().round(2)

,Interest_Rate
count,50000.00
mean,68.77
std,451.60
min,1.00
25%,8.00
50%,13.00
75%,20.00
max,5799.00


In [33]:
df.loc[df['Interest_Rate']>35 , 'Interest_Rate']=np.nan

In [34]:
df['Interest_Rate'] = df.groupby('Customer_ID')['Interest_Rate'].transform(lambda x: x.ffill().bfill())
df['Interest_Rate'].fillna(df['Interest_Rate'].median(), inplace=True)

In [35]:
df['Num_Credit_Card'].describe().to_frame().round().astype(int)

,Num_Credit_Card
count,50000
mean,23
std,129
min,0
25%,4
50%,5
75%,7
max,1499


In [36]:
df[df['Num_Credit_Card'] == 0].groupby('Name').size().sort_values(ascending=False).to_frame()

,0
Name,
Bartuneks,4
Huffstuttert,4
Singhj,4
arbara Lewish,4


In [37]:
df.loc[df['Num_Credit_Card'] > 11, 'Num_Credit_Card'] = np.nan

In [38]:
df['Num_Credit_Card'] = df.groupby('Customer_ID')['Num_Credit_Card'].transform(lambda x: x.ffill().bfill())
df['Num_Credit_Card'].fillna(df['Num_Credit_Card'].median(), inplace=True)

In [39]:
df['Num_Credit_Card'].describe().round().astype(int).to_frame()

,Num_Credit_Card
count,50000
mean,6
std,2
min,0
25%,4
50%,5
75%,7
max,11


In [40]:
df['Num_Credit_Card'] = df['Num_Credit_Card'].astype(int)

In [41]:
df['Num_of_Loan'].unique()[:20]

array(['4', '1', '3', '1381', '-100', '0', '2', '7', '5', '6', '5_', '8',
       '1_', '2_', '6_', '4_', '9', '0_', '7_', '965'], dtype=object)

In [42]:
df['Num_of_Loan'] = df['Num_of_Loan'].astype(str).str.replace(r'[^0-9.-]', '', regex=True)

df['Num_of_Loan'] = pd.to_numeric(df['Num_of_Loan'],errors='coerce')

In [43]:
df.loc[df['Num_of_Loan'] > 10, 'Num_of_Loan'] = np.nan

In [44]:
df['Num_of_Loan'] = df.groupby('Customer_ID')['Num_of_Loan'].transform(lambda x: x.ffill().bfill())
df['Num_of_Loan'].fillna(df['Num_of_Loan'].median(), inplace=True)

In [45]:
df['Num_of_Loan'] = df['Num_of_Loan'].astype(int)
df['Num_of_Loan'].dtype

dtype('int64')

In [46]:
df['Num_of_Loan'].describe().round(2).to_frame()

,Num_of_Loan
count,50000.00
mean,-0.57
std,20.35
min,-100.00
25%,1.00
50%,3.00
75%,5.00
max,9.00


In [47]:
df['Type_of_Loan'].value_counts().to_frame().nlargest(20, columns='count')

,count
Type_of_Loan,
Not Specified,704
Credit-Builder Loan,640
Personal Loan,636
Debt Consolidation Loan,632
Student Loan,620
Payday Loan,600
Mortgage Loan,588
Auto Loan,576
Home Equity Loan,568


In [48]:
df.loc[df['Type_of_Loan'].isna(), 'Type_of_Loan'] = 'Not Specified'

In [49]:
df.loc[df['Delay_from_due_date'] < 0 , 'Delay_from_due_date'] = 0

In [50]:
df['Delay_from_due_date'].fillna(0,inplace = True)

In [51]:
df['Num_of_Delayed_Payment'].value_counts().to_frame().sample(10, random_state=50)

,count
Num_of_Delayed_Payment,
3794,1
2412,1
1703,1
975,1
211,1
2819,1
2497,1
459,1
3097,1


In [52]:
df['Num_of_Delayed_Payment'] 

0          7
1          9
2          4
3          5
4          1
        ... 
49995     25
49996    NaN
49997      5
49998     6_
49999      5
Name: Num_of_Delayed_Payment, Length: 50000, dtype: object

In [53]:
df['Num_of_Delayed_Payment'] = df['Num_of_Delayed_Payment'].astype(str).str.replace(r'[^0-9.]', '', regex=True)

df['Num_of_Delayed_Payment'] = pd.to_numeric(df['Num_of_Delayed_Payment'])

In [54]:
df['Num_of_Delayed_Payment'].describe().round(2)

count    46502.00
mean        30.91
std        221.51
min          0.00
25%          9.00
50%         14.00
75%         18.00
max       4399.00
Name: Num_of_Delayed_Payment, dtype: float64

In [55]:
df.loc[df['Num_of_Delayed_Payment'] > 30, 'Num_of_Delayed_Payment'] = np.nan

In [56]:
df['Num_of_Delayed_Payment'] = df.groupby('Customer_ID')['Num_of_Delayed_Payment'].transform(lambda x: x.ffill().bfill())
df['Num_of_Delayed_Payment'].fillna(df['Num_of_Delayed_Payment'].median(), inplace=True)

In [57]:
df['Changed_Credit_Limit'].value_counts().to_frame()

,count
Changed_Credit_Limit,
_,1059
11.5,70
11.32,63
7.01,60
7.35,60
...,...
-0.6099999999999999,1
21.61,1
12.010000000000002,1


In [58]:
df['Changed_Credit_Limit'] = df['Changed_Credit_Limit'].astype(str).str.replace(r'[^0-9.]', '', regex=True)

df['Changed_Credit_Limit'] = pd.to_numeric(df['Changed_Credit_Limit'])

In [59]:
df['Changed_Credit_Limit'].value_counts().to_frame()

,count
Changed_Credit_Limit,
11.50,70
11.32,63
7.35,60
7.01,60
10.06,57
...,...
1.78,1
30.91,1
27.79,1


In [60]:
df['Changed_Credit_Limit'].fillna(df['Changed_Credit_Limit'].median(), inplace=True)

In [61]:
df['Changed_Credit_Limit'].describe().to_frame().round(2)

,Changed_Credit_Limit
count,50000.00
mean,10.44
std,6.58
min,0.00
25%,5.48
50%,9.41
75%,14.60
max,36.65


In [62]:
df.loc[df['Num_Credit_Inquiries'] > 17, 'Num_Credit_Inquiries'] = np.nan

In [63]:
df['Num_Credit_Inquiries'] = df.groupby('Customer_ID')['Num_Credit_Inquiries'].transform(lambda x: x.ffill().bfill())
df['Num_Credit_Inquiries'].fillna(df['Num_Credit_Inquiries'].median(), inplace=True)

In [64]:
df['Credit_Mix'].value_counts().to_frame()

,count
Credit_Mix,
Standard,18379
Good,12260
_,9805
Bad,9556


In [65]:
df['Credit_Mix'] = df['Credit_Mix'].replace('_', np.nan)

In [66]:
df['Credit_Mix'] = df.groupby('Customer_ID')['Credit_Mix'].transform(lambda x: x.ffill().bfill())
df['Credit_Mix'].fillna(df['Credit_Mix'].mode()[0], inplace=True)

In [67]:
df['Credit_Mix'].describe().to_frame()

,Credit_Mix
count,50000
unique,3
top,Standard
freq,22980


In [68]:
df['Outstanding_Debt'].value_counts().to_frame().sample(5, random_state=12)

,count
Outstanding_Debt,
4845.24,4
326.6,4
4896.0_,1
831.37,4
1663.52,4


In [69]:
df['Outstanding_Debt'].astype(str).str.contains('_').sum()

np.int64(491)

In [70]:
df['Outstanding_Debt'] = df['Outstanding_Debt'].astype(str).str.replace(r'[^0-9.]', '', regex=True)

df['Outstanding_Debt'] = pd.to_numeric(df['Outstanding_Debt'])

In [71]:
df['Outstanding_Debt'].value_counts().to_frame().sample(5)

,count
Outstanding_Debt,
1483.23,4
2803.61,4
156.14,4
2216.25,4
1519.71,4


In [72]:
df['Credit_Utilization_Ratio'].describe().to_frame().round(3)

,Credit_Utilization_Ratio
count,50000.000
mean,32.280
std,5.106
min,20.510
25%,28.061
50%,32.280
75%,36.469
max,48.541


In [73]:
df['Credit_History_Age'].value_counts().to_frame().nlargest(10, columns='count')

,count
Credit_History_Age,
20 Years and 1 Months,254
16 Years and 1 Months,254
18 Years and 7 Months,252
19 Years and 7 Months,252
18 Years and 6 Months,250
16 Years and 6 Months,248
19 Years and 1 Months,242
18 Years and 1 Months,241
16 Years and 7 Months,238


In [74]:
df['Credit_History_Age'].sample(10)

48532    24 Years and 4 Months
11216     7 Years and 3 Months
42329    14 Years and 5 Months
28487     6 Years and 9 Months
16085    24 Years and 7 Months
9757                       NaN
12156    17 Years and 4 Months
15870    19 Years and 5 Months
42392    5 Years and 11 Months
23598    21 Years and 6 Months
Name: Credit_History_Age, dtype: object

In [75]:
df['Credit_History_Age'] = df.groupby('Customer_ID')['Credit_History_Age'].transform(lambda x: x.ffill().bfill())
df['Credit_History_Age'].fillna(df['Credit_History_Age'].mode()[0], inplace=True)

In [76]:
df['Credit_History_Years'] = df['Credit_History_Age'].str.extract(r'(\d+)\s+Years').astype(int)

In [77]:
df['Credit_History_Years'].describe().to_frame().astype(int)

,Credit_History_Years
count,50000
mean,18
std,8
min,0
25%,12
50%,18
75%,25
max,34


In [78]:
df.drop('Credit_History_Years', axis=1, inplace=True)

In [79]:
df['Payment_of_Min_Amount'].value_counts()

Payment_of_Min_Amount
Yes    26158
No     17849
NM      5993
Name: count, dtype: int64

In [80]:
df[df['Total_EMI_per_month'] > 8000]['Name'].sample(5, random_state=41)

13778    Poornima Guptaz
44246                 Se
36875        Donny Kwoka
32704            Camposb
24829     Ronald Groverj
Name: Name, dtype: object

In [81]:
df[df['Name'] == 'Camposb'].iloc[:, 18:24].style.background_gradient(subset='Total_EMI_per_month')

,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly
32704,933.670000,30.227112,10 Years and 0 Months,Yes,54858.000000,90.04617734692921
32705,933.670000,32.430652,10 Years and 1 Months,NM,92.506250,409.97294782738123
32706,933.670000,30.668950,10 Years and 2 Months,Yes,92.506250,223.0299511205567
32707,933.670000,40.225432,10 Years and 3 Months,Yes,92.506250,450.99783708888987


In [82]:
df[df['Name'] == 'Ken Willsw'].iloc[:, 18:24].style.background_gradient(subset='Total_EMI_per_month')

,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly
41956,2297.860000,27.064830,7 Years and 0 Months,Yes,129.454940,432.6254289488672
41957,2297.860000,38.144799,7 Years and 1 Months,Yes,129.454940,203.31904770638843
41958,2297.860000,40.090893,7 Years and 2 Months,Yes,129.454940,123.15334908068371
41959,2297.860000,24.715109,7 Years and 3 Months,Yes,129.454940,54.30059912410813


In [83]:
df.loc[df['Total_EMI_per_month'] > 2000, 'Total_EMI_per_month'] = np.nan

In [84]:
df['Total_EMI_per_month'] = df.groupby('Customer_ID')['Total_EMI_per_month'].transform(lambda x: x.ffill().bfill())
df['Total_EMI_per_month'].fillna(df['Total_EMI_per_month'].median(), inplace=True)

In [85]:
df['Amount_invested_monthly'].describe()

count         47729
unique        45450
top       __10000__
freq           2175
Name: Amount_invested_monthly, dtype: object

In [86]:
df['Amount_invested_monthly'].value_counts().to_frame().nlargest(10, columns='count')

,count
Amount_invested_monthly,
__10000__,2175
0.0,106
236.64268203272135,1
160.0977166999812,1
320.4566446914704,1
51.9149815110778,1
245.26696212811387,1
138.22965938784515,1
127.74327149411123,1


In [87]:
df['Amount_invested_monthly'] = df['Amount_invested_monthly'].replace('__10000__', np.nan)

In [88]:
df['Amount_invested_monthly'] = df['Amount_invested_monthly'].astype(float)

In [89]:
df['Amount_invested_monthly'] = df.groupby('Customer_ID')['Amount_invested_monthly'].transform(lambda x: x.ffill().bfill())
df['Amount_invested_monthly'].fillna(df['Amount_invested_monthly'].median(), inplace=True)

In [90]:
df['Amount_invested_monthly'].describe().to_frame().round()

,Amount_invested_monthly
count,50000.0
mean,195.0
std,196.0
min,0.0
25%,72.0
50%,129.0
75%,237.0
max,1908.0


In [91]:
df['Payment_Behaviour'].describe()

count                              50000
unique                                 7
top       Low_spent_Small_value_payments
freq                               12694
Name: Payment_Behaviour, dtype: object

In [92]:
df['Payment_Behaviour'].value_counts().to_frame()

,count
Payment_Behaviour,
Low_spent_Small_value_payments,12694
High_spent_Medium_value_payments,8922
High_spent_Large_value_payments,6844
Low_spent_Medium_value_payments,6837
High_spent_Small_value_payments,5651
Low_spent_Large_value_payments,5252
!@9#%8,3800


In [93]:
df[df['Payment_Behaviour'] == '!@9#%8'][['Name', 'Occupation', 'Payment_Behaviour']].sample(5, random_state=11)

,Name,Occupation,Payment_Behaviour
15722,Foo Yunp,Teacher,!@9#%8
16606,"""John ODonnell""e",Scientist,!@9#%8
11301,Doeringy,Media_Manager,!@9#%8
23963,Lucia Mutikanik,Journalist,!@9#%8
19616,Tetsushi Kajimotoq,Doctor,!@9#%8


In [94]:
df['Payment_Behaviour'] = df['Payment_Behaviour'].replace('!@9#%8', np.nan)

In [95]:
df['Payment_Behaviour'] = df.groupby('Customer_ID')['Payment_Behaviour'].transform(lambda x: x.ffill().bfill())
df['Payment_Behaviour'].fillna(df['Payment_Behaviour'].mode()[0], inplace=True)

In [96]:
df['Payment_Behaviour'].value_counts().to_frame()

,count
Payment_Behaviour,
Low_spent_Small_value_payments,13681
High_spent_Medium_value_payments,9668
High_spent_Large_value_payments,7425
Low_spent_Medium_value_payments,7412
High_spent_Small_value_payments,6134
Low_spent_Large_value_payments,5680


In [97]:
df['Monthly_Balance'].describe()

count                                49438
unique                               49433
top       __-333333333333333333333333333__
freq                                     6
Name: Monthly_Balance, dtype: object

In [98]:
df.loc[df['Monthly_Balance'] == '__-333333333333333333333333333__', 'Monthly_Balance'] = np.nan
df['Monthly_Balance'] = pd.to_numeric(df['Monthly_Balance'], errors='coerce')

In [99]:
df['Monthly_Balance'] = df.groupby('Customer_ID')['Monthly_Balance'].transform(lambda x: x.ffill().bfill())
df['Monthly_Balance'].fillna(df['Monthly_Balance'].median(), inplace=True)

In [100]:
df['Monthly_Balance'].describe().to_frame().round(2)

,Monthly_Balance
count,50000.00
mean,403.06
std,214.09
min,0.10
25%,270.34
50%,337.25
75%,471.39
max,1606.52


In [101]:
df['Annual_Income'].describe().round(0).astype(int).to_frame()

,Annual_Income
count,50000
mean,166334
std,1351965
min,7006
25%,19453
50%,37578
75%,72817
max,24137255


In [102]:
df['Log_Annual_Income'] = np.log1p(df['Annual_Income'])

In [103]:
df['Log_Monthly_Inhand_Salary'] = np.log1p(df['Monthly_Inhand_Salary'])

In [104]:
df['Monthly_Inhand_Salary'].describe().round(0).astype(int).to_frame()

,Monthly_Inhand_Salary
count,50000
mean,4181
std,3173
min,304
25%,1624
50%,3082
75%,5936
max,15205


In [105]:
df['Outstanding_Debt'].describe().round(0).astype(int).to_frame()

,Outstanding_Debt
count,50000
mean,1426
std,1155
min,0
25%,566
50%,1166
75%,1946
max,4998


In [106]:
df['Log_Outstanding_Debt'] = np.log1p(df['Outstanding_Debt'])

In [107]:
df['Outstanding_Debt'].describe().round(0).astype(int).to_frame()

,Outstanding_Debt
count,50000
mean,1426
std,1155
min,0
25%,566
50%,1166
75%,1946
max,4998


In [108]:
df['Log_Outstanding_Debt'].describe().round(0).astype(int).to_frame()

,Log_Outstanding_Debt
count,50000
mean,7
std,1
min,0
25%,6
50%,7
75%,8
max,9


In [109]:
df['Log_Total_EMI_per_month'] = np.log1p(df['Total_EMI_per_month'])

In [110]:
df['Log_Amount_invested_monthly'] = np.log1p(df['Amount_invested_monthly'])

In [111]:
df['Log_Amount_invested_monthly'].describe().round(0).astype(int).to_frame()

,Log_Amount_invested_monthly
count,50000
mean,5
std,1
min,0
25%,4
50%,5
75%,5
max,8


In [112]:
df['Log_Monthly_Balance'] = np.log1p(df['Monthly_Balance'])

In [113]:
df['Log_Monthly_Balance'].describe().round(0).astype(int).to_frame()

,Log_Monthly_Balance
count,50000
mean,6
std,1
min,0
25%,6
50%,6
75%,6
max,7


In [114]:
df['Log_Income_Ratio'] = np.log1p(df['Monthly_Inhand_Salary'] * 12) - df['Log_Annual_Income']

In [115]:
df['Log_Income_Ratio'].describe().to_frame().round(3)

,Log_Income_Ratio
count,50000.000
mean,-0.065
std,0.562
min,-7.928
25%,-0.051
50%,-0.005
75%,0.036
max,1.445


In [116]:
all_loans_split = df['Type_of_Loan'].str.split(',\s*and\s*|\s*,\s*')

flat_loan_list = [loan.strip() for sublist in all_loans_split for loan in sublist]

unique_loan_types = set(flat_loan_list)

In [117]:
df['Type_of_Loan'].value_counts().to_frame().nlargest(40, columns='count')

,count
Type_of_Loan,
Not Specified,6408
Credit-Builder Loan,640
Personal Loan,636
Debt Consolidation Loan,632
Student Loan,620
Payday Loan,600
Mortgage Loan,588
Auto Loan,576
Home Equity Loan,568


In [118]:
df['Type_of_Loan_List'] = df['Type_of_Loan'].str.split(',\s*and\s*|\s*,\s*')

In [119]:
df['Type_of_Loan_List'].value_counts().to_frame().nlargest(15, columns='count') 

,count
Type_of_Loan_List,
[Not Specified],6408
[Credit-Builder Loan],640
[Personal Loan],636
[Debt Consolidation Loan],632
[Student Loan],620
[Payday Loan],600
[Mortgage Loan],588
[Auto Loan],576
[Home Equity Loan],568


In [120]:
mlb = MultiLabelBinarizer()
loan_dummies = pd.DataFrame(
    mlb.fit_transform(df['Type_of_Loan_List']),
    columns=mlb.classes_,
    index=df.index
)

loan_dummies.head()

,Auto Loan,Credit-Builder Loan,Debt Consolidation Loan,Home Equity Loan,Mortgage Loan,Not Specified,Payday Loan,Personal Loan,Student Loan
0,1,1,0,1,0,0,0,1,0
1,1,1,0,1,0,0,0,1,0
2,1,1,0,1,0,0,0,1,0
3,1,1,0,1,0,0,0,1,0
4,0,1,0,0,0,0,0,0,0


In [121]:
joblib.dump(mlb, "mlb.pkl")
print("MultiLabelBinarizer Saved Successfully")

MultiLabelBinarizer Saved Successfully


In [122]:
loan_counts_from_dummies = loan_dummies.sum().sort_values(ascending=False)

loan_counts_df = loan_counts_from_dummies.to_frame(name="count")
loan_counts_df

,count
Not Specified,21544
Payday Loan,15972
Credit-Builder Loan,15864
Home Equity Loan,15700
Mortgage Loan,15680
Personal Loan,15552
Debt Consolidation Loan,15520
Student Loan,15520
Auto Loan,15280


In [123]:
df = pd.concat([df, loan_dummies], axis=1)

In [124]:
df.columns.to_frame(index=False)

,0
0,ID
1,Customer_ID
2,Month
3,Name
4,Age
5,Occupation
6,Annual_Income
7,Monthly_Inhand_Salary
8,Num_Bank_Accounts
9,Num_Credit_Card


In [125]:
kmeans = KMeans(n_clusters=3, random_state=42)
df['Debt_Level'] = kmeans.fit_predict(df[['Outstanding_Debt']])

In [126]:
stats = df.groupby('Debt_Level')['Outstanding_Debt'].agg(
    min_value='min',
    max_value='max',
    mean_value='mean',
    median_value='median',
    std_value='std',
    count='count',
    q25=lambda x: x.quantile(0.25),
    q75=lambda x: x.quantile(0.75)
).reset_index()

stats.style.background_gradient()

,Debt_Level,min_value,max_value,mean_value,median_value,std_value,count,q25,q75
0,0,1163.470000,2823.830000,1758.659655,1549.335000,490.596623,19032,1344.450000,2177.660000
1,1,2825.560000,4998.070000,3899.886246,3869.250000,628.510640,6020,3374.260000,4445.870000
2,2,0.230000,1163.400000,575.713205,565.260000,333.282760,24948,294.330000,859.420000


In [127]:
df["Debt_Level"] = df["Debt_Level"].map({
    0:'Low',
    1:'Medium',
    2:'High'
})

In [128]:
df['Credit_History_by_Months'] = (df['Credit_History_Age']
                                         .str.extract(r'(\d+)\s+Years\s+and\s+(\d+)\s+Months')
                                         .astype(float)
                                         .apply(lambda x: x[0] * 12 + x[1], axis=1))

In [129]:
df['Credit_History_by_Months'].describe().to_frame().round().astype(int)

,Credit_History_by_Months
count,50000
mean,227
std,100
min,10
25%,150
50%,225
75%,307
max,408


In [130]:
df['Credit_History_by_Months'] = df['Credit_History_by_Months'].astype(int) 

In [131]:
df_engineered = df.drop(['ID', 'Customer_ID', 'Name', 'Annual_Income',  'Monthly_Inhand_Salary', 'Type_of_Loan', 'Type_of_Loan_List', 'Outstanding_Debt', 'Credit_History_Age', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance'], axis=1)

In [132]:
df_engineered.columns.to_frame(index=False)

,0
0,Month
1,Age
2,Occupation
3,Num_Bank_Accounts
4,Num_Credit_Card
5,Interest_Rate
6,Num_of_Loan
7,Delay_from_due_date
8,Num_of_Delayed_Payment
9,Changed_Credit_Limit


In [133]:
ordered_columns = [
    "Month",
    "Age",
    "Occupation",
    "Log_Annual_Income",
    "Log_Monthly_Inhand_Salary",
    "Log_Income_Ratio",
    "Num_Bank_Accounts",
    "Num_Credit_Card",
    "Interest_Rate",
    "Changed_Credit_Limit",
    "Num_Credit_Inquiries",
    "Credit_Mix",
    "Credit_Utilization_Ratio",
    "Num_of_Loan",
    "Auto Loan",
    "Credit-Builder Loan",
    "Debt Consolidation Loan",
    "Home Equity Loan",
    "Mortgage Loan",
    "Not Specified",
    "Payday Loan",
    "Personal Loan",
    "Student Loan",
    "Debt_Level",
    "Delay_from_due_date",
    "Num_of_Delayed_Payment",
    "Payment_of_Min_Amount",
    "Payment_Behaviour",
    "Log_Outstanding_Debt",
    "Log_Total_EMI_per_month",
    "Log_Amount_invested_monthly",
    "Log_Monthly_Balance",
    "Credit_History_by_Months"
    
]

df_engineered = df_engineered[ordered_columns]

In [134]:
df_engineered

,Month,Age,Occupation,Log_Annual_Income,Log_Monthly_Inhand_Salary,Log_Income_Ratio,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Credit_Utilization_Ratio,Num_of_Loan,Auto Loan,Credit-Builder Loan,Debt Consolidation Loan,Home Equity Loan,Mortgage Loan,Not Specified,Payday Loan,Personal Loan,Student Loan,Debt_Level,Delay_from_due_date,Num_of_Delayed_Payment,Payment_of_Min_Amount,Payment_Behaviour,Log_Outstanding_Debt,Log_Total_EMI_per_month,Log_Amount_invested_monthly,Log_Monthly_Balance,Credit_History_by_Months
0,September,23.0,Scientist,9.858235,7.509797,0.135967,3,4,3.0,11.27,4.0,Good,35.030402,4,1,1,0,1,0,0,0,1,0,High,3,7.0,No,Low_spent_Small_value_payments,6.698243,3.923456,5.470768,5.232534,273
1,October,24.0,Scientist,9.858235,7.509797,0.135967,3,4,3.0,13.27,4.0,Good,33.053114,4,1,1,0,1,0,0,0,1,0,High,3,9.0,No,High_spent_Medium_value_payments,6.698243,3.923456,3.111975,5.892870,274
2,November,24.0,Scientist,9.858235,7.509797,0.135967,3,4,3.0,12.27,4.0,Good,33.811894,4,1,1,0,1,0,0,0,1,0,High,0,4.0,No,Low_spent_Medium_value_payments,6.698243,3.923456,5.005515,5.582275,274
3,December,24.0,Scientist,9.858235,7.509797,0.135967,3,4,3.0,11.27,4.0,Good,32.430559,4,1,1,0,1,0,0,0,1,0,High,4,5.0,No,High_spent_Medium_value_payments,6.698243,3.923456,3.690940,5.843042,276
4,September,28.0,Teacher,10.458775,8.019279,0.045109,2,4,6.0,5.42,5.0,Good,25.926822,1,0,1,0,0,0,0,0,0,0,High,3,1.0,No,High_spent_Large_value_payments,6.406929,2.986501,3.705835,6.186822,327
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,December,29.0,Architect,9.903682,7.565745,0.146495,10,8,29.0,18.31,12.0,Bad,34.780553,5,1,0,0,0,1,0,0,1,1,Medium,33,25.0,Yes,Low_spent_Small_value_payments,8.181077,4.126566,4.993735,5.622354,78
49996,September,25.0,Mechanic,10.587341,8.119820,0.017112,4,6,7.0,11.50,7.0,Good,27.758522,2,1,0,0,0,0,0,0,0,1,High,20,5.0,NM,Low_spent_Small_value_payments,6.221345,3.586404,5.206438,6.017119,383
49997,October,25.0,Mechanic,10.587341,8.119820,0.017112,4,6,7.0,13.50,7.0,Good,36.858542,2,1,0,0,0,0,0,0,0,1,High,23,5.0,No,Low_spent_Large_value_payments,6.221345,3.586404,5.206438,5.860006,384
49998,November,25.0,Mechanic,10.587341,8.119820,0.017112,4,6,7.0,11.50,7.0,Good,39.139840,2,1,0,0,0,0,0,0,0,1,High,21,6.0,No,High_spent_Small_value_payments,6.221345,3.586404,4.591057,6.140399,385


In [135]:
import json
with open("ordered_columns.json", "w") as f:
    json.dump(ordered_columns, f)

In [136]:
df_engineered.columns.to_frame(index=False, name='Dataset Columns After Ordering')

,Dataset Columns After Ordering
0,Month
1,Age
2,Occupation
3,Log_Annual_Income
4,Log_Monthly_Inhand_Salary
5,Log_Income_Ratio
6,Num_Bank_Accounts
7,Num_Credit_Card
8,Interest_Rate
9,Changed_Credit_Limit


In [137]:
def summarize_dataframe(df):
    summary = []
    for col in df.columns:
        series = df[col]
        if isinstance(series, pd.DataFrame):
            series = series.iloc[:,0]
        data_type = series.dtype
        missing = series.isna().sum()
        missing_pct = round((missing / len(df)) * 100, 2)
        unique_vals = series.nunique()
        summary.append([col, data_type, missing, missing_pct, unique_vals])
    return pd.DataFrame(summary, columns=['Column', 'Data Type', 'Missing', 'Missing %', 'Unique Values'])

In [138]:
df_info = summarize_dataframe(df_engineered)
df_info.style.background_gradient(cmap='Purples')

,Column,Data Type,Missing,Missing %,Unique Values
0,Month,object,0,0.000000,4
1,Age,float64,0,0.000000,43
2,Occupation,object,0,0.000000,15
3,Log_Annual_Income,float64,0,0.000000,12989
4,Log_Monthly_Inhand_Salary,float64,0,0.000000,12793
5,Log_Income_Ratio,float64,0,0.000000,13288
6,Num_Bank_Accounts,int64,0,0.000000,11
7,Num_Credit_Card,int64,0,0.000000,12
8,Interest_Rate,float64,0,0.000000,34
9,Changed_Credit_Limit,float64,0,0.000000,3473


In [139]:
df_engineered.columns

Index(['Month', 'Age', 'Occupation', 'Log_Annual_Income',
       'Log_Monthly_Inhand_Salary', 'Log_Income_Ratio', 'Num_Bank_Accounts',
       'Num_Credit_Card', 'Interest_Rate', 'Changed_Credit_Limit',
       'Num_Credit_Inquiries', 'Credit_Mix', 'Credit_Utilization_Ratio',
       'Num_of_Loan', 'Auto Loan', 'Credit-Builder Loan',
       'Debt Consolidation Loan', 'Home Equity Loan', 'Mortgage Loan',
       'Not Specified', 'Payday Loan', 'Personal Loan', 'Student Loan',
       'Debt_Level', 'Delay_from_due_date', 'Num_of_Delayed_Payment',
       'Payment_of_Min_Amount', 'Payment_Behaviour', 'Log_Outstanding_Debt',
       'Log_Total_EMI_per_month', 'Log_Amount_invested_monthly',
       'Log_Monthly_Balance', 'Credit_History_by_Months'],
      dtype='object')

In [140]:
df_engineered.head(10)

,Month,Age,Occupation,Log_Annual_Income,Log_Monthly_Inhand_Salary,Log_Income_Ratio,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Credit_Utilization_Ratio,Num_of_Loan,Auto Loan,Credit-Builder Loan,Debt Consolidation Loan,Home Equity Loan,Mortgage Loan,Not Specified,Payday Loan,Personal Loan,Student Loan,Debt_Level,Delay_from_due_date,Num_of_Delayed_Payment,Payment_of_Min_Amount,Payment_Behaviour,Log_Outstanding_Debt,Log_Total_EMI_per_month,Log_Amount_invested_monthly,Log_Monthly_Balance,Credit_History_by_Months
0,September,23.0,Scientist,9.858235,7.509797,0.135967,3,4,3.0,11.27,4.0,Good,35.030402,4,1,1,0,1,0,0,0,1,0,High,3,7.0,No,Low_spent_Small_value_payments,6.698243,3.923456,5.470768,5.232534,273
1,October,24.0,Scientist,9.858235,7.509797,0.135967,3,4,3.0,13.27,4.0,Good,33.053114,4,1,1,0,1,0,0,0,1,0,High,3,9.0,No,High_spent_Medium_value_payments,6.698243,3.923456,3.111975,5.892870,274
2,November,24.0,Scientist,9.858235,7.509797,0.135967,3,4,3.0,12.27,4.0,Good,33.811894,4,1,1,0,1,0,0,0,1,0,High,0,4.0,No,Low_spent_Medium_value_payments,6.698243,3.923456,5.005515,5.582275,274
3,December,24.0,Scientist,9.858235,7.509797,0.135967,3,4,3.0,11.27,4.0,Good,32.430559,4,1,1,0,1,0,0,0,1,0,High,4,5.0,No,High_spent_Medium_value_payments,6.698243,3.923456,3.690940,5.843042,276
4,September,28.0,Teacher,10.458775,8.019279,0.045109,2,4,6.0,5.42,5.0,Good,25.926822,1,0,1,0,0,0,0,0,0,0,High,3,1.0,No,High_spent_Large_value_payments,6.406929,2.986501,3.705835,6.186822,327
5,October,28.0,Teacher,10.458775,8.019279,0.045109,2,4,6.0,5.42,5.0,Good,30.116600,1,0,1,0,0,0,0,0,0,0,High,3,3.0,No,Low_spent_Large_value_payments,6.406929,2.986501,5.531916,5.718195,328
6,November,28.0,Teacher,10.458775,8.019279,0.045109,2,4,6.0,5.42,5.0,Good,30.996424,1,0,1,0,0,0,0,0,0,0,High,3,3.0,No,High_spent_Large_value_payments,6.406929,2.986501,4.299733,6.116559,329
7,December,28.0,Teacher,10.458775,8.019279,0.045109,2,4,6.0,7.42,5.0,Good,33.875167,1,0,1,0,0,0,0,0,0,0,High,3,2.0,No,High_spent_Large_value_payments,6.406929,2.986501,5.040417,6.046066,330
8,September,35.0,Engineer,11.871744,9.408225,0.021313,1,5,8.0,7.10,3.0,Good,35.229707,3,1,0,0,0,0,1,0,0,0,Low,8,3.0,No,Low_spent_Medium_value_payments,7.173199,5.513398,5.987717,6.751366,221
9,October,35.0,Engineer,11.871744,9.408225,0.021313,1,5,8.0,2.10,3.0,Good,35.685836,3,1,0,0,0,0,1,0,0,0,Low,6,3.0,No,Low_spent_Large_value_payments,7.173199,5.513398,6.119451,6.670911,222


# Features & Target Split

In [141]:
X = df_engineered.drop('Credit_Mix', axis=1)
y = df_engineered['Credit_Mix']

In [142]:
y.value_counts()

Credit_Mix
Standard    22980
Good        15168
Bad         11852
Name: count, dtype: int64

In [143]:
# Encode target
label_encoder_target = LabelEncoder()
y_encoded = label_encoder_target.fit_transform(y)

In [144]:
# 2. Encoding Pipelines
ordinal_cols = ['Payment_of_Min_Amount']
nominal_cols = ['Month', 'Occupation', 'Payment_Behaviour']

In [145]:
# Ordinal
ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder())
])

# Nominal
nominal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

In [146]:
# Preprocessor
preprocessor = ColumnTransformer([
    ('ord', ordinal_pipeline, ordinal_cols),
    ('ohn', nominal_pipeline, nominal_cols)
], remainder='passthrough')

# Train & Test Split

In [147]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y)

In [148]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)
encoded_feature_names = (
    ordinal_cols +
    list(preprocessor.named_transformers_['ohn'].named_steps['encoder'].get_feature_names_out(nominal_cols)) +
    [col for col in X.columns if col not in ordinal_cols + nominal_cols]
)

In [149]:
X_train.columns.tolist()

['Month',
 'Age',
 'Occupation',
 'Log_Annual_Income',
 'Log_Monthly_Inhand_Salary',
 'Log_Income_Ratio',
 'Num_Bank_Accounts',
 'Num_Credit_Card',
 'Interest_Rate',
 'Changed_Credit_Limit',
 'Num_Credit_Inquiries',
 'Credit_Utilization_Ratio',
 'Num_of_Loan',
 'Auto Loan',
 'Credit-Builder Loan',
 'Debt Consolidation Loan',
 'Home Equity Loan',
 'Mortgage Loan',
 'Not Specified',
 'Payday Loan',
 'Personal Loan',
 'Student Loan',
 'Debt_Level',
 'Delay_from_due_date',
 'Num_of_Delayed_Payment',
 'Payment_of_Min_Amount',
 'Payment_Behaviour',
 'Log_Outstanding_Debt',
 'Log_Total_EMI_per_month',
 'Log_Amount_invested_monthly',
 'Log_Monthly_Balance',
 'Credit_History_by_Months']

In [150]:
ordinal_cols

['Payment_of_Min_Amount']

In [151]:
nominal_cols

['Month', 'Occupation', 'Payment_Behaviour']

In [152]:
all_feature_names_unique = list(dict.fromkeys(encoded_feature_names))

In [153]:
X_train_encoded = pd.DataFrame(X_train_encoded, columns=all_feature_names_unique, index=X_train.index)
X_test_encoded  = pd.DataFrame(X_test_encoded, columns=all_feature_names_unique, index=X_test.index)

In [154]:
encoded_feature_names = X_train_encoded.columns.tolist()

In [155]:
joblib.dump(preprocessor, 'preprocessor.pkl')
print('✅ Encoder pipeline Saved Successfully')

joblib.dump(ordinal_pipeline, 'oridinal_encoder.pkl')
print('✅ Oridinal Pipline Saved Successfully')

joblib.dump(nominal_pipeline, 'onehot_encoder.pkl')
print('✅ OneHot Pipline Saved Successfully')

joblib.dump(label_encoder_target, 'target_encoder.pkl')
print('✅ Traget Encoder Saved Successfully')

joblib.dump(encoded_feature_names, "encoded_feature_names.pkl")
print('✅ Encoded Feature Names Saved Successfully')

✅ Encoder pipeline Saved Successfully
✅ Oridinal Pipline Saved Successfully
✅ OneHot Pipline Saved Successfully
✅ Traget Encoder Saved Successfully
✅ Encoded Feature Names Saved Successfully


In [156]:
X_train_encoded.columns.tolist()

['Payment_of_Min_Amount',
 'Month_December',
 'Month_November',
 'Month_October',
 'Month_September',
 'Occupation_Accountant',
 'Occupation_Architect',
 'Occupation_Developer',
 'Occupation_Doctor',
 'Occupation_Engineer',
 'Occupation_Entrepreneur',
 'Occupation_Journalist',
 'Occupation_Lawyer',
 'Occupation_Manager',
 'Occupation_Mechanic',
 'Occupation_Media_Manager',
 'Occupation_Musician',
 'Occupation_Scientist',
 'Occupation_Teacher',
 'Occupation_Writer',
 'Payment_Behaviour_High_spent_Large_value_payments',
 'Payment_Behaviour_High_spent_Medium_value_payments',
 'Payment_Behaviour_High_spent_Small_value_payments',
 'Payment_Behaviour_Low_spent_Large_value_payments',
 'Payment_Behaviour_Low_spent_Medium_value_payments',
 'Payment_Behaviour_Low_spent_Small_value_payments',
 'Age',
 'Log_Annual_Income',
 'Log_Monthly_Inhand_Salary',
 'Log_Income_Ratio',
 'Num_Bank_Accounts',
 'Num_Credit_Card',
 'Interest_Rate',
 'Changed_Credit_Limit',
 'Num_Credit_Inquiries',
 'Credit_Utiliza

In [157]:
mapping = {'Low':0, 'Medium':1 , 'High':2}
X_train_encoded['Debt_Level'] = X_train_encoded['Debt_Level'].map(mapping)
X_test_encoded['Debt_Level'] = X_test_encoded['Debt_Level'].map(mapping)

In [158]:
X_train_encoded = X_train_encoded.astype(float)
X_test_encoded = X_test_encoded.astype(float)

In [159]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

In [160]:
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [161]:
X_train_scaled

array([[-2.02870155, -0.57492539,  1.7272146 , ..., -0.5977383 ,
        -0.54425592, -0.33051993],
       [-0.58370597, -0.57492539, -0.57896685, ...,  1.94444449,
         1.71798205,  0.09044923],
       [-0.58370597, -0.57492539, -0.57896685, ...,  0.95170458,
        -0.61184102, -0.16012765],
       ...,
       [-0.58370597, -0.57492539, -0.57896685, ..., -0.27342913,
        -0.45772097,  1.38342596],
       [-0.58370597, -0.57492539,  1.7272146 , ...,  1.05822991,
         0.04008557,  1.68411822],
       [-0.58370597,  1.73935614, -0.57896685, ..., -0.37357262,
         1.23513558,  0.76199529]])

# Modeling

In [162]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(random_state=42, 
                               class_weight='balanced',
                               n_estimators=300,
                               verbose=1,
                               max_features='sqrt',
                               n_jobs=-1)

model_rf.fit(X_train_encoded, y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:   29.5s
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed:  1.9min
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  2.7min finished


RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=42, verbose=1)

In [163]:
y_pred_train_rf = model_rf.predict(X_train_encoded)
print(classification_report(y_train, y_pred_train_rf, target_names=label_encoder_target.classes_))

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    1.2s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    5.7s
[Parallel(n_jobs=2)]: Done 300 out of 300 | elapsed:    9.3s finished


              precision    recall  f1-score   support

         Bad       1.00      1.00      1.00      9482
        Good       1.00      1.00      1.00     12134
    Standard       1.00      1.00      1.00     18384

    accuracy                           1.00     40000
   macro avg       1.00      1.00      1.00     40000
weighted avg       1.00      1.00      1.00     40000



In [164]:
y_pred_test_rf = model_rf.predict(X_test_encoded)
print(classification_report(y_test, y_pred_test_rf, target_names=label_encoder_target.classes_))

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.4s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    1.9s


              precision    recall  f1-score   support

         Bad       0.98      0.99      0.99      2370
        Good       0.98      0.99      0.99      3034
    Standard       0.99      0.98      0.98      4596

    accuracy                           0.99     10000
   macro avg       0.98      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



[Parallel(n_jobs=2)]: Done 300 out of 300 | elapsed:    3.3s finished


In [165]:
X_train_encoded = pd.DataFrame(X_train_encoded, columns=encoded_feature_names)
X_test_encoded  = pd.DataFrame(X_test_encoded, columns=encoded_feature_names)

In [166]:
importances_rf = model_rf.feature_importances_

feature_importance_rf = pd.DataFrame({
    'Feature': X_train_encoded.columns,
    'Importance': importances_rf
}).sort_values(by="Importance", ascending=False)

feature_importance_rf.head(20).style.background_gradient()

,Feature,Importance
32,Interest_Rate,0.158198
48,Num_of_Delayed_Payment,0.127388
0,Payment_of_Min_Amount,0.106307
49,Log_Outstanding_Debt,0.095413
47,Delay_from_due_date,0.072386
30,Num_Bank_Accounts,0.069329
36,Num_of_Loan,0.064166
53,Credit_History_by_Months,0.047948
33,Changed_Credit_Limit,0.047405
34,Num_Credit_Inquiries,0.037396


In [167]:
model_lgb = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=12,
    num_leaves=30,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    force_col_wise=True
)


model_lgb.fit(
    X_train_encoded, y_train,
    eval_set=[(X_train_encoded, y_train), (X_test_encoded, y_test)],
    eval_metric='multi_logloss',
    callbacks=[
        early_stopping(stopping_rounds=50),  
        log_evaluation(period=50)
        ]
)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Total Bins 2846
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 54
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score -1.439484
[LightGBM] [Info] Start training from score -1.192868
[LightGBM] [Info] Start training from score -0.777399
Training until validation scores don't improve for 50 rounds
[50]	training's multi_logloss: 0.144257	valid_1's multi_logloss: 0.151784
[100]	training's multi_logloss: 0.0875545	valid_1's multi_logloss: 0.10316
[150]	training's multi_logloss: 0.0641476	valid_1's multi_logloss: 0.0840752
[200]	training's multi_logloss: 0.0476803	valid_1's multi_logloss: 0.0698179
[250]	training's multi_logloss: 0.0363981	valid_1's multi_logloss: 0.0601562
[300]	training's multi_logloss: 0.0279525	valid_1's multi_logloss: 0.0522561
[350]	training's multi_logloss: 0

LGBMClassifier(colsample_bytree=0.8, force_col_wise=True, learning_rate=0.05,
               max_depth=12, n_estimators=800, n_jobs=-1, num_leaves=30,
               random_state=42, subsample=0.8)

In [168]:
y_pred_train = model_lgb.predict(X_train_encoded)
print(classification_report(y_train, y_pred_train, target_names=label_encoder_target.classes_))

              precision    recall  f1-score   support

         Bad       1.00      1.00      1.00      9482
        Good       1.00      1.00      1.00     12134
    Standard       1.00      1.00      1.00     18384

    accuracy                           1.00     40000
   macro avg       1.00      1.00      1.00     40000
weighted avg       1.00      1.00      1.00     40000



In [174]:
y_pred_test = model_rf.predict(X_test_encoded)

print(classification_report(
    y_test,
    y_pred_test,
    target_names=label_encoder_target.classes_
))

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    7.7s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:   14.1s
[Parallel(n_jobs=2)]: Done 300 out of 300 | elapsed:   17.0s finished


              precision    recall  f1-score   support

         Bad       0.98      0.99      0.99      2370
        Good       0.98      0.99      0.99      3034
    Standard       0.99      0.98      0.98      4596

    accuracy                           0.99     10000
   macro avg       0.98      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



In [175]:
feature_importance_rf.head(20)

,Feature,Importance
32,Interest_Rate,0.158198
48,Num_of_Delayed_Payment,0.127388
0,Payment_of_Min_Amount,0.106307
49,Log_Outstanding_Debt,0.095413
47,Delay_from_due_date,0.072386
30,Num_Bank_Accounts,0.069329
36,Num_of_Loan,0.064166
53,Credit_History_by_Months,0.047948
33,Changed_Credit_Limit,0.047405
34,Num_Credit_Inquiries,0.037396


In [169]:
importances_lgb= model_lgb.feature_importances_

feature_importance_lgb = pd.DataFrame({
    'Feature': X_train_encoded.columns,
    'Importance': importances_lgb
}).sort_values(by="Importance", ascending=False)

feature_importance_lgb.head(20).style.background_gradient()

,Feature,Importance
49,Log_Outstanding_Debt,5877
53,Credit_History_by_Months,4748
50,Log_Total_EMI_per_month,4739
29,Log_Income_Ratio,4684
33,Changed_Credit_Limit,4409
28,Log_Monthly_Inhand_Salary,4284
26,Age,4038
47,Delay_from_due_date,3986
32,Interest_Rate,3936
27,Log_Annual_Income,3672


In [170]:
X_train_encoded.columns.tolist()

['Payment_of_Min_Amount',
 'Month_December',
 'Month_November',
 'Month_October',
 'Month_September',
 'Occupation_Accountant',
 'Occupation_Architect',
 'Occupation_Developer',
 'Occupation_Doctor',
 'Occupation_Engineer',
 'Occupation_Entrepreneur',
 'Occupation_Journalist',
 'Occupation_Lawyer',
 'Occupation_Manager',
 'Occupation_Mechanic',
 'Occupation_Media_Manager',
 'Occupation_Musician',
 'Occupation_Scientist',
 'Occupation_Teacher',
 'Occupation_Writer',
 'Payment_Behaviour_High_spent_Large_value_payments',
 'Payment_Behaviour_High_spent_Medium_value_payments',
 'Payment_Behaviour_High_spent_Small_value_payments',
 'Payment_Behaviour_Low_spent_Large_value_payments',
 'Payment_Behaviour_Low_spent_Medium_value_payments',
 'Payment_Behaviour_Low_spent_Small_value_payments',
 'Age',
 'Log_Annual_Income',
 'Log_Monthly_Inhand_Salary',
 'Log_Income_Ratio',
 'Num_Bank_Accounts',
 'Num_Credit_Card',
 'Interest_Rate',
 'Changed_Credit_Limit',
 'Num_Credit_Inquiries',
 'Credit_Utiliza

In [171]:
df_engineered.columns.tolist()

['Month',
 'Age',
 'Occupation',
 'Log_Annual_Income',
 'Log_Monthly_Inhand_Salary',
 'Log_Income_Ratio',
 'Num_Bank_Accounts',
 'Num_Credit_Card',
 'Interest_Rate',
 'Changed_Credit_Limit',
 'Num_Credit_Inquiries',
 'Credit_Mix',
 'Credit_Utilization_Ratio',
 'Num_of_Loan',
 'Auto Loan',
 'Credit-Builder Loan',
 'Debt Consolidation Loan',
 'Home Equity Loan',
 'Mortgage Loan',
 'Not Specified',
 'Payday Loan',
 'Personal Loan',
 'Student Loan',
 'Debt_Level',
 'Delay_from_due_date',
 'Num_of_Delayed_Payment',
 'Payment_of_Min_Amount',
 'Payment_Behaviour',
 'Log_Outstanding_Debt',
 'Log_Total_EMI_per_month',
 'Log_Amount_invested_monthly',
 'Log_Monthly_Balance',
 'Credit_History_by_Months']